In [9]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


drive.mount() = Connecting my Colab notebook to my Google Drive so I can access the dataset and save files.

In [10]:
import pandas as pd

# Loading the dataset
df = pd.read_csv('/content/drive/MyDrive/CSC4110_G07/data/weatherAUS.csv')

# Keeping  an untouched copy called df_raw
df_raw = df.copy()
print("Data loaded successfully.")

Data loaded successfully.


This cell pulls the weather dataset from our shared Drive using Pandas, and saves a backup copy called df_raw so we always have the original, messy data untouched for reference.



In [11]:
# 1. To display column names and data types
print("--- df.info() ---")
df.info()

# 2. Counting missing values in each column
print("\n--- Missing Values per Column ---")
print(df.isnull().sum())

# Counting duplicate rows
print(f"\nDuplicate rows: {df.duplicated().sum()}")

--- df.info() ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 142193 entries, 0 to 142192
Data columns (total 24 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           142193 non-null  object 
 1   Location       142193 non-null  object 
 2   MinTemp        141556 non-null  float64
 3   MaxTemp        141871 non-null  float64
 4   Rainfall       140787 non-null  float64
 5   Evaporation    81350 non-null   float64
 6   Sunshine       74377 non-null   float64
 7   WindGustDir    132863 non-null  object 
 8   WindGustSpeed  132923 non-null  float64
 9   WindDir9am     132180 non-null  object 
 10  WindDir3pm     138415 non-null  object 
 11  WindSpeed9am   140845 non-null  float64
 12  WindSpeed3pm   139563 non-null  float64
 13  Humidity9am    140419 non-null  float64
 14  Humidity3pm    138583 non-null  float64
 15  Pressure9am    128179 non-null  float64
 16  Pressure3pm    128212 non-null  float64
 17  Cloud9am   

This cell is our Data Inspection. It does three things:
1. df.info() shows us the columns, data types, and initial row count.
2. isnull().sum() counts exactly how many missing values are in each column so we know what to clean.
3. duplicated().sum() checks if there are any exact duplicate rows in the dataset (which there aren't).

In [12]:
# 3. To display summary statistics
print("--- df.describe() ---")
display(df.describe())

# 4. Showing value counts of classification target (RainTomorrow)
print("\n--- RainTomorrow Counts ---")
print(df['RainTomorrow'].value_counts(dropna=False))

--- df.describe() ---


,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RISK_MM
count,141556.000000,141871.000000,140787.000000,81350.000000,74377.000000,132923.000000,140845.000000,139563.000000,140419.000000,138583.000000,128179.000000,128212.000000,88536.000000,85099.000000,141289.000000,139467.000000,142193.000000
mean,12.186400,23.226784,2.349974,5.469824,7.624853,39.984292,14.001988,18.637576,68.843810,51.482606,1017.653758,1015.258204,4.437189,4.503167,16.987509,21.687235,2.360682
std,6.403283,7.117618,8.465173,4.188537,3.781525,13.588801,8.893337,8.803345,19.051293,20.797772,7.105476,7.036677,2.887016,2.720633,6.492838,6.937594,8.477969
min,-8.500000,-4.800000,0.000000,0.000000,0.000000,6.000000,0.000000,0.000000,0.000000,0.000000,980.500000,977.100000,0.000000,0.000000,-7.200000,-5.400000,0.000000
25%,7.600000,17.900000,0.000000,2.600000,4.900000,31.000000,7.000000,13.000000,57.000000,37.000000,1012.900000,1010.400000,1.000000,2.000000,12.300000,16.600000,0.000000
50%,12.000000,22.600000,0.000000,4.800000,8.500000,39.000000,13.000000,19.000000,70.000000,52.000000,1017.600000,1015.200000,5.000000,5.000000,16.700000,21.100000,0.000000
75%,16.800000,28.200000,0.800000,7.400000,10.600000,48.000000,19.000000,24.000000,83.000000,66.000000,1022.400000,1020.000000,7.000000,7.000000,21.600000,26.400000,0.800000
max,33.900000,48.100000,371.000000,145.000000,14.500000,135.000000,130.000000,87.000000,100.000000,100.000000,1041.000000,1039.600000,9.000000,9.000000,40.200000,46.700000,371.000000



--- RainTomorrow Counts ---
RainTomorrow
No     110316
Yes     31877
Name: count, dtype: int64


This cell gives us a high-level summary of the numbers and our target. df.describe() shows the min, max, and average for all numeric columns so we can spot weird values (like negative rainfall). The value_counts at the bottom shows us our class imbalance—that there are way more 'No' days than 'Yes' rain days.
describe() shows numeric ranges (min/max) to spot outliers. value_counts shows the target class imbalance (RainTomorrow: ~77% No, ~23% Yes).

In [13]:
# 1. Convert Date to a date type and drop rows where RainTomorrow is missing
df['Date'] = pd.to_datetime(df['Date'])
df = df.dropna(subset=['RainTomorrow'])
print(f"Rows after dropping missing RainTomorrow: {len(df)}")

# 2. Report the percentage missing per column; drop columns > 35% missing
missing_percent = df.isnull().mean() * 100
print("\nPercentage missing per column:")
print(missing_percent[missing_percent > 0].sort_values(ascending=False))

# Identify and drop columns > 35% missing
cols_to_drop = missing_percent[missing_percent > 35].index.tolist()
print(f"\nDropping columns (>35% missing): {cols_to_drop}")
df = df.drop(columns=cols_to_drop)

# 3. Fill numeric gaps with the median of the same Location and month; categorical with mode
# Create a temporary month column for grouping
df['Month'] = df['Date'].dt.month

# Fill numeric columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    df[col] = df.groupby(['Location', 'Month'])[col].transform(lambda x: x.fillna(x.median()))

# Fill categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if col != 'RainTomorrow': # Don't fill the target
        df[col] = df.groupby(['Location', 'Month'])[col].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))

# Drop the temporary Month column
df = df.drop(columns=['Month'])
print("\nMissing values after median/mode imputation:")
print(df.isnull().sum().sum(), "total missing values remaining.")

Rows after dropping missing RainTomorrow: 142193

Percentage missing per column:
Sunshine         47.692924
Evaporation      42.789026
Cloud3pm         40.152469
Cloud9am         37.735332
Pressure9am       9.855619
Pressure3pm       9.832411
WindDir9am        7.041838
WindGustDir       6.561504
WindGustSpeed     6.519308
WindDir3pm        2.656952
Humidity3pm       2.538803
Temp3pm           1.917113
WindSpeed3pm      1.849599
Humidity9am       1.247600
Rainfall          0.988797
RainToday         0.988797
WindSpeed9am      0.948007
Temp9am           0.635756
MinTemp           0.447983
MaxTemp           0.226453
dtype: float64

Dropping columns (>35% missing): ['Evaporation', 'Sunshine', 'Cloud9am', 'Cloud3pm']


/tmp/ipykernel_2816/2803844531.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df.groupby(['Location', 'Month'])[col].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))



Missing values after median/mode imputation:
35504 total missing values remaining.


This is our main cleaning cell. It does four things:
1. Changes the Date column to a real date format and drops any rows missing our target (RainTomorrow).
2. Calculates the missing percentage and drops the 4 columns that are over 35% missing.
3. Creates a temporary 'Month' column so we can group the data by Location and Month. We then fill missing numbers with the median and missing text with the mode.
4. Drops the temporary 'Month' column to finish.

In [14]:
# 4. Apply IQR rule to WindGustSpeed, cap it, and keep heavy Rainfall
outlier_col = "WindGustSpeed"
Q1 = df[outlier_col].quantile(0.25)
Q3 = df[outlier_col].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Cap the outliers
df[outlier_col] = df[outlier_col].clip(lower=lower, upper=upper)
print(f"Capped {outlier_col} at lower={lower:.2f} and upper={upper:.2f}")

# We keep heavy Rainfall values because extreme rainfall is a critical real-world event
# that the Mvua Alert system must predict and warn farmers/fishers about.

# 5. Fix wrong data types and drop RISK_MM (leakage)
df['RainTomorrow'] = df['RainTomorrow'].map({'No': 0, 'Yes': 1})
df['RainToday'] = df['RainToday'].map({'No': 0, 'Yes': 1})

# Drop RISK_MM because it is tomorrow's rainfall (target leakage)
if 'RISK_MM' in df.columns:
    df = df.drop(columns=['RISK_MM'])
    print("Dropped RISK_MM to prevent leakage.")

# Call the result df_stage1
df_stage1 = df.copy()
print("Cleaning complete. df_stage1 is ready.")

Capped WindGustSpeed at lower=5.50 and upper=73.50
Dropped RISK_MM to prevent leakage.
Cleaning complete. df_stage1 is ready.


This cell handles outliers and data types:
1. We apply the IQR rule to WindGustSpeed to find the upper and lower limits, and cap any extreme winds so they don't skew the model.
2. We leave heavy rainfall alone because extreme rain is exactly what the alert system needs to predict.
3. We map our Yes/No targets to 1s and 0s.
4. Finally, we drop the RISK_MM column. This is a critical step because RISK_MM is literally tomorrow's rainfall amount—if we kept it, the model would 'cheat' (data leakage) and just look at tomorrow's rain to predict tomorrow's rain.

In [15]:
# Fill any remaining gaps globally just in case a station+month combo was entirely empty
# (This ensures the dataset is 100% clean for the next tasks)
for col in df_stage1.columns:
    if df_stage1[col].dtype == 'object' or df_stage1[col].dtype == 'int64':
        # Fill categorical with the overall mode
        if not df_stage1[col].mode().empty:
            df_stage1[col] = df_stage1[col].fillna(df_stage1[col].mode()[0])
    else:
        # Fill numeric with the overall median
        df_stage1[col] = df_stage1[col].fillna(df_stage1[col].median())

print(f"Total missing values remaining: {df_stage1.isnull().sum().sum()}")

Total missing values remaining: 0


This is a final safety-net cell. Sometimes a specific weather station might have missing data for an entire month, meaning the Location+Month grouping didn't catch it. This loop goes through every column and fills any leftover gaps with the overall median (for numbers) or mode (for text). This guarantees we have exactly zero missing values before we move on to modeling.

In [16]:
# Take the first 8 values of Pressure9am, and set the third one to missing on a copy
pressure_sample = df_stage1['Pressure9am'].head(8).copy()
pressure_sample.iloc[2] = None
print("Data with missing value:")
print(pressure_sample)

# Show the median imputation by hand in the Markdown cell below
# Then fill it with fillna
pressure_filled = pressure_sample.fillna(pressure_sample.median())
print("\nData after fillna (median):")
print(pressure_filled)

Data with missing value:
0    1007.7
1    1010.6
2       NaN
3    1017.6
4    1010.8
5    1009.2
6    1009.6
7    1013.4
Name: Pressure9am, dtype: float64

Data after fillna (median):
0    1007.7
1    1010.6
2    1010.6
3    1017.6
4    1010.8
5    1009.2
6    1009.6
7    1013.4
Name: Pressure9am, dtype: float64


**Exam Skill: Median Imputation by Hand**

The first 8 values of Pressure9am (with the 3rd set to missing) are:
[1007.7, 1010.6, NaN, 1017.6, 1010.8, 1009.2, 1009.6, 1013.4]

To find the median by hand, we sort the remaining 7 values from lowest to highest:
[1007.7, 1009.2, 1009.6, 1010.6, 1010.8, 1013.4, 1017.6]

The middle value (4th position) is **1010.6**. Therefore, the median imputation fills the missing value with 1010.6.

**Explanation of the fillna code:**
The line `pressure_sample.fillna(pressure_sample.median())` calculates the median of the existing values in the column and replaces every NaN (missing) value with that specific median number.